In [2]:
! pip install featuretools


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.9/587.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.2/215.2 kB 17.8 MB/s eta 0:00:00


In [3]:
### Carga de librerias
import pandas as pd
import featuretools as ft

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
### Cargamos  el data set
data = ft.demo.load_mock_customer()

In [5]:
## Podemos ver que es un diccionario de tablas
print(data['customers'])
print(data['products'])
print(data['sessions'])
print(data['transactions'])

   customer_id zip_code           join_date   birthday
0            1    60091 2011-04-17 10:48:33 1994-07-18
1            2    13244 2012-04-15 23:31:04 1986-08-18
2            3    13244 2011-08-13 15:42:34 2003-11-21
3            4    60091 2011-04-08 20:08:14 2006-08-15
4            5    60091 2010-07-17 05:27:50 1984-07-28
  product_id brand
0          1     B
1          2     B
2          3     B
3          4     B
4          5     A
    session_id  customer_id   device       session_start
0            1            2  desktop 2014-01-01 00:00:00
1            2            5   mobile 2014-01-01 00:17:20
2            3            4   mobile 2014-01-01 00:28:10
3            4            1   mobile 2014-01-01 00:44:25
4            5            4   mobile 2014-01-01 01:11:30
5            6            1   tablet 2014-01-01 01:23:25
6            7            3   tablet 2014-01-01 01:39:40
7            8            4   tablet 2014-01-01 01:55:55
8            9            1  desktop 2014-0

In [6]:
## Separamos en cada dataset
df_customers = data["customers"]
df_customers.set_index("customer_id", inplace=True)
df_customers.head()

,zip_code,join_date,birthday
customer_id,,,
1,60091,2011-04-17 10:48:33,1994-07-18
2,13244,2012-04-15 23:31:04,1986-08-18
3,13244,2011-08-13 15:42:34,2003-11-21
4,60091,2011-04-08 20:08:14,2006-08-15
5,60091,2010-07-17 05:27:50,1984-07-28


In [7]:
df_products = data["products"]
df_products.set_index("product_id", inplace=True)
df_products.head()

,brand
product_id,
1,B
2,B
3,B
4,B
5,A


In [8]:
df_sessions = data["sessions"]
df_sessions.set_index("session_id", inplace=True)
df_sessions.head()

,customer_id,device,session_start
session_id,,,
1,2,desktop,2014-01-01 00:00:00
2,5,mobile,2014-01-01 00:17:20
3,4,mobile,2014-01-01 00:28:10
4,1,mobile,2014-01-01 00:44:25
5,4,mobile,2014-01-01 01:11:30


In [9]:
df_transactions = data["transactions"]
df_transactions.set_index("transaction_id", inplace=True)
df_transactions.head()

,session_id,transaction_time,product_id,amount
transaction_id,,,,
2,1,2014-01-01 00:00:00,5,127.64
495,1,2014-01-01 00:01:05,2,109.48
341,1,2014-01-01 00:02:10,3,95.06
308,1,2014-01-01 00:03:15,4,78.92
271,1,2014-01-01 00:04:20,3,31.54


In [ ]:
## Crea el objeto EntitySet
# Un EntitySet es un contenedor que agrupa varios DataFrames relacionados entre sí, como una mini base de datos en memoria.
es = ft.EntitySet()

es = es.add_dataframe(dataframe_name="customers", dataframe=df_customers, index="customer_id")
es = es.add_dataframe(dataframe_name="products", dataframe=df_products, index="product_id")
es = es.add_dataframe(dataframe_name="transactions", dataframe=df_transactions, index="transaction_id")
es = es.add_dataframe(dataframe_name="sessions", dataframe=df_sessions, index="session_id")

es = es.add_relationship("products", "product_id", "transactions", "product_id")
es = es.add_relationship("sessions", "session_id", "transactions", "session_id")
es = es.add_relationship("customers", "customer_id", "sessions", "customer_id")

es

/home/juanma/anaconda3/envs/dm_clase15/lib/python3.11/site-packages/featuretools/entityset/entityset.py:1733: UserWarning: index customer_id not found in dataframe, creating new integer column
  warnings.warn(
/home/juanma/anaconda3/envs/dm_clase15/lib/python3.11/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/juanma/anaconda3/envs/dm_clase15/lib/python3.11/site-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/home/juanma/anaconda3/envs/dm_clase15/lib/python3.11/site-packages/featuretools/entityset/entityset.py:1733: UserWarning: index product_id not found in dataframe, creating new integer col

Entityset: None
  DataFrames:
    customers [Rows: 5, Columns: 4]
    products [Rows: 5, Columns: 2]
    transactions [Rows: 500, Columns: 5]
    sessions [Rows: 35, Columns: 4]
  Relationships:
    transactions.product_id -> products.product_id
    transactions.session_id -> sessions.session_id
    sessions.customer_id -> customers.customer_id

In [ ]:
es.plot()

ImportError: Please install graphviz to plot. (See https://featuretools.alteryx.com/en/stable/install.html#installing-graphviz for details)

feature_matrix → el DataFrame final. Una fila por cliente, una columna por feature generada. Esto es lo que le pasarías a un modelo.
feature_defs → la lista de features con su definición. Sirve para reproducir exactamente las mismas transformaciones sobre datos nuevos en producción.

In [ ]:
#feature_matrix, feature_defs = ft.dfs(entityset=es, target_dataframe_name="products")
feature_matrix, feature_defs = ft.dfs(entityset=es, target_dataframe_name="customers")
#feature_matrix, feature_defs = ft.dfs(entityset=es, target_dataframe_name="transactions")
#feature_matrix, feature_defs = ft.dfs(entityset=es, target_dataframe_name="sessions")

In [ ]:
feature_matrix

,zip_code,COUNT(sessions),MODE(sessions.device),NUM_UNIQUE(sessions.device),COUNT(transactions),MAX(transactions.amount),MEAN(transactions.amount),MIN(transactions.amount),SKEW(transactions.amount),STD(transactions.amount),...,STD(sessions.MIN(transactions.amount)),STD(sessions.SKEW(transactions.amount)),STD(sessions.SUM(transactions.amount)),SUM(sessions.MAX(transactions.amount)),SUM(sessions.MEAN(transactions.amount)),SUM(sessions.MIN(transactions.amount)),SUM(sessions.SKEW(transactions.amount)),SUM(sessions.STD(transactions.amount)),MODE(transactions.sessions.device),NUM_UNIQUE(transactions.sessions.device)
customer_id,,,,,,,,,,,,,,,,,,,,,
0,60091,0,NaN,<NA>,0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.00,0.000000,0.00,0.000000,0.000000,NaN,<NA>
1,13244,8,mobile,3,119,149.15,75.148655,5.73,0.123116,45.163495,...,5.739770,0.444126,316.388722,1108.98,599.570158,85.21,0.389205,352.566101,mobile,3
2,13244,7,desktop,3,67,146.81,81.233134,5.81,-0.160628,40.371822,...,19.517969,0.483145,418.124271,830.04,486.661571,109.32,0.001909,238.898839,tablet,3
3,60091,6,desktop,3,82,148.86,82.132927,8.73,-0.060535,42.633775,...,17.521535,0.366322,236.944472,811.24,498.781825,129.60,-1.628207,237.283142,desktop,3
4,60091,8,mobile,3,123,149.95,71.882439,6.29,0.165638,42.276191,...,7.111911,0.351386,297.486182,1130.76,582.678756,98.37,1.281989,344.551882,mobile,3


In [ ]:
feature_defs

[<Feature: zip_code>,
 <Feature: COUNT(sessions)>,
 <Feature: MODE(sessions.device)>,
 <Feature: NUM_UNIQUE(sessions.device)>,
 <Feature: COUNT(transactions)>,
 <Feature: MAX(transactions.amount)>,
 <Feature: MEAN(transactions.amount)>,
 <Feature: MIN(transactions.amount)>,
 <Feature: SKEW(transactions.amount)>,
 <Feature: STD(transactions.amount)>,
 <Feature: SUM(transactions.amount)>,
 <Feature: DAY(birthday)>,
 <Feature: DAY(join_date)>,
 <Feature: MONTH(birthday)>,
 <Feature: MONTH(join_date)>,
 <Feature: WEEKDAY(birthday)>,
 <Feature: WEEKDAY(join_date)>,
 <Feature: YEAR(birthday)>,
 <Feature: YEAR(join_date)>,
 <Feature: MAX(sessions.COUNT(transactions))>,
 <Feature: MAX(sessions.MEAN(transactions.amount))>,
 <Feature: MAX(sessions.MIN(transactions.amount))>,
 <Feature: MAX(sessions.SKEW(transactions.amount))>,
 <Feature: MAX(sessions.STD(transactions.amount))>,
 <Feature: MAX(sessions.SUM(transactions.amount))>,
 <Feature: MEAN(sessions.COUNT(transactions))>,
 <Feature: MEAN(ses

In [ ]:
## Parte 2 pycaret

In [ ]:
import pycaret

In [ ]:
from pycaret import datasets
df = pycaret.datasets.get_data('insurance')

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [ ]:
from pycaret.regression import RegressionExperiment
exp = pycaret.regression.RegressionExperiment()

# Inicializa el entorno en pycaret y crea el proceso de transformación para preparar los datos para el modelado y la implementación.
reg = exp.setup(data=df, target="charges")

,Description,Value
0,Session id,1815
1,Target,charges
2,Target type,Regression
3,Original data shape,"(1338, 7)"
4,Transformed data shape,"(1338, 10)"
5,Transformed train set shape,"(936, 10)"
6,Transformed test set shape,"(402, 10)"
7,Numeric features,3
8,Categorical features,3
9,Preprocess,True


In [ ]:
best = reg.compare_models()

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,2650.9711,22264701.4523,4655.3870,0.8396,0.4462,0.3132,0.0330
lightgbm,Light Gradient Boosting Machine,2948.3624,24116727.3632,4852.1854,0.8261,0.5413,0.3766,67.7450
rf,Random Forest Regressor,2800.6397,24494694.0826,4886.0392,0.8231,0.4761,0.3360,0.0600
et,Extra Trees Regressor,2758.8252,27013972.0790,5128.2603,0.8046,0.4883,0.3226,0.0500
ada,AdaBoost Regressor,4181.8987,28113285.2994,5279.6687,0.7964,0.6193,0.7010,0.0200
lar,Least Angle Regression,4113.5188,36779505.6132,6038.4537,0.7404,0.5830,0.4258,0.0190
llar,Lasso Least Angle Regression,4113.6362,36776562.6558,6038.2034,0.7404,0.5867,0.4258,0.0160
br,Bayesian Ridge,4120.2469,36778062.8378,6038.3671,0.7404,0.5846,0.4272,0.0190
ridge,Ridge Regression,4126.9118,36777890.0489,6038.3833,0.7404,0.5799,0.4286,0.0180
lasso,Lasso Regression,4113.6487,36776620.3904,6038.2083,0.7404,0.5866,0.4259,0.1330


In [ ]:
## Otro dataset

In [ ]:
df = pycaret.datasets.get_data("hepatitis")

,Class,AGE,SEX,STEROID,ANTIVIRALS,FATIGUE,MALAISE,ANOREXIA,LIVER BIG,LIVER FIRM,SPLEEN PALPABLE,SPIDERS,ASCITES,VARICES,BILIRUBIN,ALK PHOSPHATE,SGOT,ALBUMIN,PROTIME,HISTOLOGY
0,0,30,2,1.0,2,2,2,2,1.0,2.0,2.0,2.0,2.0,2.0,1.0,85.0,18.0,4.0,NaN,1
1,0,50,1,1.0,2,1,2,2,1.0,2.0,2.0,2.0,2.0,2.0,0.9,135.0,42.0,3.5,NaN,1
2,0,78,1,2.0,2,1,2,2,2.0,2.0,2.0,2.0,2.0,2.0,0.7,96.0,32.0,4.0,NaN,1
3,0,31,1,NaN,1,2,2,2,2.0,2.0,2.0,2.0,2.0,2.0,0.7,46.0,52.0,4.0,80.0,1
4,0,34,1,2.0,2,2,2,2,2.0,2.0,2.0,2.0,2.0,2.0,1.0,NaN,200.0,4.0,NaN,1


In [ ]:
from pycaret.classification import ClassificationExperiment

exp = pycaret.classification.ClassificationExperiment()
clf = exp.setup(data=df, target="Class", imputation_type="iterative", categorical_iterative_imputer="rf", numeric_iterative_imputer="rf", iterative_imputation_iters=5)

,Description,Value
0,Session id,7798
1,Target,Class
2,Target type,Binary
3,Original data shape,"(154, 20)"
4,Transformed data shape,"(154, 20)"
5,Transformed train set shape,"(107, 20)"
6,Transformed test set shape,"(47, 20)"
7,Numeric features,19
8,Rows with missing values,48.1%
9,Preprocess,True


In [ ]:
clf.X_train.head()

,AGE,SEX,STEROID,ANTIVIRALS,FATIGUE,MALAISE,ANOREXIA,LIVER BIG,LIVER FIRM,SPLEEN PALPABLE,SPIDERS,ASCITES,VARICES,BILIRUBIN,ALK PHOSPHATE,SGOT,ALBUMIN,PROTIME,HISTOLOGY
138,45,1,2.0,1,2,2,2,2.0,2.0,2.0,2.0,2.0,2.0,1.3,85.0,44.0,4.2,85.0,2
133,38,1,1.0,2,2,2,2,2.0,1.0,2.0,2.0,2.0,2.0,0.4,243.0,49.0,3.8,90.0,2
42,33,1,2.0,2,2,2,2,2.0,2.0,2.0,2.0,2.0,2.0,1.0,46.0,90.0,4.4,60.0,1
23,42,1,2.0,2,2,2,2,2.0,2.0,2.0,2.0,2.0,2.0,0.9,60.0,63.0,4.7,47.0,1
135,51,1,2.0,2,2,2,2,1.0,1.0,2.0,1.0,2.0,2.0,0.8,NaN,33.0,4.5,NaN,2


In [ ]:
best = clf.compare_models(sort='AUC')

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
nb,Naive Bayes,0.7691,0.8854,0.8333,0.4967,0.6071,0.4655,0.5058,1.5170
rf,Random Forest Classifier,0.8518,0.8778,0.5333,0.5833,0.5333,0.4719,0.4880,1.5820
ada,Ada Boost Classifier,0.7864,0.8694,0.5000,0.3900,0.4105,0.3103,0.3262,1.5930
et,Extra Trees Classifier,0.8509,0.8625,0.5333,0.5833,0.5267,0.4670,0.4869,1.6020
lr,Logistic Regression,0.8336,0.8451,0.4667,0.4833,0.4633,0.3914,0.3980,1.4990
lightgbm,Light Gradient Boosting Machine,0.7945,0.8354,0.4833,0.3733,0.4071,0.3120,0.3236,10.8880
ridge,Ridge Classifier,0.8336,0.8326,0.4167,0.5500,0.4633,0.3857,0.3962,1.5430
lda,Linear Discriminant Analysis,0.8145,0.8215,0.4167,0.5167,0.4433,0.3536,0.3657,1.5750
gbc,Gradient Boosting Classifier,0.8155,0.8194,0.4833,0.4667,0.4600,0.3684,0.3777,1.5670
svm,SVM - Linear Kernel,0.6882,0.7743,0.6333,0.3772,0.4173,0.2705,0.3227,1.5550
